### Links and Resources
| Concept | Reference |
|---|---|
| [`pandas_udf`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.pandas_udf.html) | API entry point |
| [Pandas UDFs guide](https://spark.apache.org/docs/latest/api/python/user_guide/sql/arrow_pandas.html) | Official guide |
| [Apache Arrow](https://arrow.apache.org/) | The serialization format |
| [`applyInPandas`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.applyInPandas.html) | Per-group DataFrame → DataFrame |
| [`mapInPandas`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.mapInPandas.html) | Per-partition iterator |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# Pandas UDFs (Vectorized)

A Pandas UDF receives **a `pandas.Series`** (or `DataFrame`) instead of one Python value at a time, runs your code on the whole batch using NumPy/pandas, and returns a `Series`. Spark moves the data via **Arrow**, the columnar format pandas already speaks — so there's no row-by-row Pickle round trip.

The win: **5–100× faster than a classic Python UDF** for anything that vectorizes naturally.

## The four flavors

| Flavor | Input | Output | Used with |
|---|---|---|---|
| **Series → Series** (Scalar) | `pd.Series` | `pd.Series` (same length) | `.select`, `.withColumn` |
| **Iterator of Series → Iterator of Series** | iterator | iterator | Long-lived state per executor |
| **Series → scalar** (Grouped Agg) | `pd.Series` | one value | `.groupBy(...).agg(...)` |
| **DataFrame → DataFrame** (Grouped Map) | `pd.DataFrame` | `pd.DataFrame` | `.groupBy(...).applyInPandas(...)` |

In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql.functions import pandas_udf

random.seed = lambda *_: None  # silence; not used
np.random.seed(0)

n = 100_000
df = spark.createDataFrame(
    pd.DataFrame({
        "id":      np.arange(n),
        "group":   np.random.choice(["A", "B", "C"], size=n),
        "value":   np.random.randn(n) * 10 + 50,
        "name":    np.random.choice(["alice", "BOB", "Charlie", "dana"], size=n),
    })
)
df.show(5)
print("rows:", n)

## 1. Scalar Pandas UDF — Series → Series

Same length in, same length out. By far the most common pattern.

In [ ]:
@pandas_udf("double")
def z_score(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()

# NOTE: this computes the mean per *batch* — see the warning below
df.select("value", z_score("value").alias("z")).show(5)

**Pitfall:** A scalar Pandas UDF gets called per **batch** (one chunk of a partition). Aggregations like `s.mean()` apply to that batch only — not the whole DataFrame. For "z-score against the overall mean" you'd compute the mean separately and subtract, or use a Grouped Agg UDF.

## 2. Iterator of Series → Iterator of Series — long-lived state

Use this when you have an expensive setup (load a model, open a connection) you only want to do **once per executor**, not once per batch.

In [ ]:
from typing import Iterator

@pandas_udf("string")
def expensive_lookup(it: Iterator[pd.Series]) -> Iterator[pd.Series]:
    # Setup runs ONCE per executor, before the first batch
    table = {"alice": "ALICE-EU", "bob": "BOB-US", "charlie": "CHARLIE-EU", "dana": "DANA-EU"}
    for s in it:
        yield s.str.lower().map(table).fillna("UNKNOWN")

df.select("name", expensive_lookup("name").alias("region")).show(5, truncate=False)

## 3. Grouped Agg Pandas UDF — Series → scalar

Inside `.groupBy(...).agg(...)`. Receives one `Series` per group, returns one value.

In [ ]:
@pandas_udf("double")
def trimmed_mean(s: pd.Series) -> float:
    # Drop top/bottom 5% then mean — not a built-in
    lo, hi = s.quantile(0.05), s.quantile(0.95)
    return s[(s >= lo) & (s <= hi)].mean()

df.groupBy("group").agg(
    F.avg("value").alias("plain_mean"),
    trimmed_mean("value").alias("trimmed_mean"),
).show()

## 4. Grouped Map — `applyInPandas`

The most powerful and most expensive form. Each group is shipped to one executor as a single `pandas.DataFrame`; you return a transformed `DataFrame` of any shape. **All rows for a group must fit in memory on one executor.**

Schema of the output is required up-front.

In [ ]:
def rank_within_group(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.copy()
    pdf["rank"] = pdf["value"].rank(method="min", ascending=False).astype("int64")
    pdf["pct"]  = pdf["value"].rank(pct=True)
    return pdf

out_schema = "id long, group string, value double, name string, rank long, pct double"
df.groupBy("group").applyInPandas(rank_within_group, schema=out_schema).show(5)

## 5. `mapInPandas` — partition-level iterator (no grouping)

Like `applyInPandas` but at the partition level — one `DataFrame` chunk at a time, no group key. Useful for stateful streaming-style logic that doesn't need a key.

In [ ]:
def add_running_id(it):
    for pdf in it:
        pdf = pdf.copy()
        pdf["row_in_partition"] = np.arange(len(pdf))
        yield pdf

df.limit(20).mapInPandas(
    add_running_id,
    schema="id long, group string, value double, name string, row_in_partition long",
).show()

## Quick benchmark — classic UDF vs Pandas UDF

`Series` math is just NumPy under the hood, so the same logic runs at near-C speed.

In [ ]:
import time
from pyspark.sql.functions import udf

@udf("double")
def py_normalize(v):
    return None if v is None else (v - 50) / 10

@pandas_udf("double")
def pd_normalize(s: pd.Series) -> pd.Series:
    return (s - 50) / 10

def bench(label, expr):
    expr.write.format("noop").mode("overwrite").save()
    t = time.time()
    expr.write.format("noop").mode("overwrite").save()
    print(f"  {label:18s} {time.time() - t:5.2f}s")

big = df.union(df).union(df)             # ~300k rows
bench("built-in",   big.select((F.col("value") - 50) / 10))
bench("Pandas UDF", big.select(pd_normalize("value")))
bench("classic UDF",big.select(py_normalize("value")))

## Memory and batch size

- Scalar / Iterator UDFs are batched. Default batch size is `spark.sql.execution.arrow.maxRecordsPerBatch = 10000`. Larger batches → less overhead but more memory per task.
- `applyInPandas` ships the *entire group* as one DataFrame. A skewed group can OOM an executor. If groups are unbalanced, salt the key or pre-aggregate.

## Type hints are part of the API

Spark introspects the type hints on your function (`pd.Series`, `Iterator[pd.Series]`, `pd.DataFrame`) to pick the right flavor. Keep them — without them, the API falls back to deprecated string-mode flavors.

## Takeaways

- Pandas UDFs eliminate the per-row JVM↔Python round trip — typical 10× win over classic UDFs.
- Use **Scalar** for column-wise math, **Iterator** when there's expensive per-executor setup, **Grouped Agg** for custom aggregations, **Grouped Map** when the result shape differs per group.
- `applyInPandas` requires the whole group to fit in one executor's memory.
- Type hints aren't optional — they pick the API flavor.
- Built-ins still beat Pandas UDFs when one exists. Use Pandas UDFs for the gap between "built-in does it" and "needs a Python library".